# Prithvi-WxC Inference

This notebook provides functionality to perform inference using the Prithv-WxC model over given time period.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# Setup

## Cloning the Prithvi Precip package

````
git clone https://github.com/simonpf/prithvi_precip
````

## Required Software

The ``environment.yml`` file in ``prithvi_precip/notebooks/prithvi_tc`` defines a conda environment containing all required software for running the inference.

````
conda create --file prithvi_precip/notebooks/prithvi_tc/environment.yml
conda activate prithvi_tc
pip install -e prithvi_precip
````

## Input Data

The following steps are required to prepare the input data for running the forecast.

### Indexing of the MERRA-2 Raw Data

This step tells the ``prithvi_precip`` package where MERRA-2 files are located on the current system.

````
pansat index /path/to/merra2/data --recursive
````

### Creating the Input Data

The inference functionality expects the following input data


````
.
├── climatology # The Prithvi-WxC climatology files including musigma_surface.nc and musigma_vertical.nc
├── input_data
│   └── dynamic
└── static
    └── static.nc # Static input data
````


#### Static Data

The static data doesn't change with the initialization time and therefore only needs to be extracted once.

```
prithvi_precip extract_static_merra_data .
```

#### Dynamic Data

The command below prepares the dynamic input data, which needs to be available for the period over which the forecasts are to be performed.
The ``prithvi_precip extract_merra_data`` command prepares the data for a given year and month.

```
mkdir input_data
prithvi_precip extract_merra_data 2024 1 input_data  --n_processes 8
...
prithvi_precip extract_merra_data 2024 12 input_data  --n_processes 8
```

#### Additional Data

The code also expects the climatology data in a folder named ``climatology`` in the parent directory of ``input_data``. 


The environment variable below should point to the folder containing the ``musigma_surface.nc`` and ``musigma_vertical.nc`` files.

In [2]:
%env PRITHVI_DATA_PATH=/data/precipfm/climatology

env: PRITHVI_DATA_PATH=/data/precipfm/climatology


## Load the model

In [3]:
from pytorch_retrieve.architectures import load_and_compile_model
from pytorch_retrieve.training import load_weights

mdl = load_and_compile_model("model.toml")
model_weights = Path("/data/prithvi_tc/prithvi.wxc.rollout.2300m.v1.pt")

load_weights(
    {"backbone": model_weights},
    mdl
)

## Inference

The function below is used to extract only winds and pressure fields from the forecast results.

In [4]:
import torch
from typing import Dict

def post_process_results(inpt: Dict[str, torch.Tensor], results: Dict[str, torch.Tensor]) -> xr.Dataset:
    """
    Extracts surface winds, surface pressure and 850 winds from forecast results.

    Args:
        inpt: The batch containing the input data.
        results: A dictionary containing the forecast results.

    Return:
        An xarray.Dataset containing the results to be stored.
    """
    static = inpt["static"]
    if static.dim() == 5:
        lats = np.rad2deg(inpt["static"][0, 0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 0, 1, 0, :].float().cpu().numpy())
    else:
        lats = np.rad2deg(inpt["static"][0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 1, 0, :].float().cpu().numpy())

    dataset = xr.Dataset({
        "latitude": (("latitude",), lats),
        "longitude": (("longitude",), lons)
    })

    pred = results["y"]
    pred = np.stack([step[:, [9, 17, 18, -18, -4]].float().cpu().numpy() for step in pred], axis=1)
    slp = pred[:, :, 0]
    u10 = pred[:, :, 1]
    v10 = pred[:, :, 2]
    u850 = pred[:, :, 3]
    v850 = pred[:, :, 4]

    dataset["slp"] = (("batch", "step", "latitude", "longitude"), slp)
    dataset["u10"] = (("batch", "step", "latitude", "longitude"), u10)
    dataset["v10"] = (("batch", "step", "latitude", "longitude"), v10)
    dataset["u850"] = (("batch", "step", "latitude", "longitude"), u850)
    dataset["v850"] = (("batch", "step", "latitude", "longitude"), v850)

    for var in ["slp", "u10", "v10", "u850", "v850"]:
        dataset[var].encoding = {"dtype": "float32", "zlib": True}
    return dataset

### Data Loader

The data loader loads the input data for the forecasts.

In [5]:
from pathlib import Path
from prithvi_precip.forecast.data_loaders import AutoregressiveForecastLoader
from torch.utils.data import DataLoader

input_data_path = Path("/data/prithvi_tc/input_data/")

# Adapt this to run forecast for longer time range.
init_times = np.arange(
    np.datetime64("2020-01-01"),
    np.datetime64("2020-02-01"),
    np.timedelta64(6, "h")
)

data_loader = AutoregressiveForecastLoader(
    input_data_path,
    init_times=init_times,
    n_steps=20, # Number of 6-hour forecast steps.
    input_time=6, # Input data time step
    center_meridionally=False,
    full_climatology=True,
)
data_loader = DataLoader(data_loader, batch_size=None, num_workers=1, collate_fn=lambda x: x)
print(f"Found input data for {len(data_loader)} forecasts.")

Found input data for 111 forecasts.


## Run the inference

In [ ]:
from prithvi_precip.forecast.runners import run_autoregressive_forecast

output_path = Path("/data/prithvi_tc/results")
output_path.mkdir(exist_ok=True)

device = "cuda:0"

run_autoregressive_forecast(
    mdl,
    data_loader,
    output_path,
    post_process_fn=post_process_results,
    device=device,
    dtype=torch.float32,
)

  8%|█████▎                                                           | 9/111 [05:51<1:06:33, 39.15s/it]